In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Optimization Solver: Uma Qiskit Function do Q-CTRL Fire Opal
> **Note:** As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via IBM Quantum Platform API). Elas estão em status de lançamento de pré-visualização e sujeitas a alterações.

## Visão geral
Com o Fire Opal Optimization Solver, você pode resolver problemas de otimização em escala utilitária em hardware quântico sem precisar de conhecimento especializado em computação quântica. Basta inserir a definição do problema em alto nível e o Solver cuida do resto. Todo o fluxo de trabalho é ciente de ruído e utiliza o [Gerenciamento de Desempenho do Fire Opal](/guides/q-ctrl-performance-management) por baixo dos panos. O Solver entrega consistentemente soluções precisas para problemas classicamente desafiadores, mesmo em escala de dispositivo completo nos maiores QPUs da IBM&reg;.

O Solver é flexível e pode ser usado para resolver problemas de otimização combinatória definidos como funções objetivo ou grafos arbitrários. Os problemas não precisam ser mapeados para a topologia do dispositivo. Tanto problemas sem restrições quanto com restrições são resolúveis, desde que as restrições possam ser formuladas como termos de penalidade. Os exemplos incluídos neste guia demonstram como resolver um problema de otimização em escala utilitária sem restrições e um com restrições, usando diferentes tipos de entrada do Solver. O primeiro exemplo envolve um problema Max-Cut definido em um grafo 3-regular com 156 nós, enquanto o segundo aborda um problema de Cobertura Mínima de Vértices com 50 nós definido por uma função de custo.

Para obter acesso ao Optimization Solver, [entre em contato com o Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrição da função
O Solver otimiza e automatiza completamente todo o algoritmo, desde a supressão de erros no nível de hardware até o mapeamento eficiente de problemas e a otimização clássica em malha fechada. Por baixo dos panos, o pipeline do Solver reduz erros em cada etapa, possibilitando o desempenho aprimorado necessário para escalar de forma significativa. O fluxo de trabalho subjacente é inspirado no Quantum Approximate Optimization Algorithm (QAOA), que é um algoritmo híbrido quântico-clássico. Para um resumo detalhado do fluxo de trabalho completo do Optimization Solver, consulte [o manuscrito publicado](https://arxiv.org/abs/2406.01743).

![Visualização do fluxo de trabalho do Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Para resolver um problema genérico com o Optimization Solver:
1. Defina seu problema como uma função objetivo, um grafo ou uma cadeia de spin `SparsePauliOp`.
2. Conecte-se à função por meio do Catálogo de Qiskit Functions.
3. Execute o problema com o Solver e recupere os resultados.
## Entradas e saídas
### Entradas
| Nome    | Tipo                    | Descrição                                                                                                                                                                                         | Obrigatório | Padrão | Exemplo |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` ou `SparsePauliOp` | Uma das representações listadas em "Formatos de problema aceitos"                                                                                                                                  | Sim | N/A   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | Nome da classe do problema; usado apenas para definições de problema em grafo e cadeia de spin, que se limitam a "maxcut" ou "spin_chain"; não é necessário para definições de problema com função objetivo arbitrária | Não      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | Nome do backend a ser usado                                                                                                                                                                          | Não  | Backend menos ocupado ao qual sua instância tem acesso    | `"ibm_fez"`      |
| options  | `dict`                  | Opções de entrada, incluindo: (Opcional) `session_id`: `str`; o comportamento padrão cria uma nova Sessão                                                                                      | Não | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**Formatos de problema aceitos:**
- Representação por expressão polinomial de uma função objetivo. Idealmente criada em Python com um objeto SymPy Poly existente e formatada como string usando [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Representação em grafo de um tipo específico de problema. O grafo deve ser criado usando a biblioteca networkx em Python e, em seguida, convertido para string usando a função networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Representação em cadeia de spin de um problema específico. A cadeia de spin deve ser representada como um objeto `SparsePauliOp`; consulte a [documentação](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) para mais detalhes.

**Backends suportados:**
Execute o código a seguir para ver a lista de backends atualmente suportados. Se o seu dispositivo não estiver listado, [entre em contato com o Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) para adicionar suporte.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**Opções:**
| Nome   | Tipo   | Descrição  | Padrão |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | ID de uma sessão do Qiskit Runtime existente  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | Uma lista de tags de job | `[]`|

### Saídas
| Nome    | Tipo | Descrição | Exemplo |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solução e metadados listados em "Conteúdo do dicionário de resultados"         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**Conteúdo do dicionário de resultados:**
| Nome    | Tipo | Descrição |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | O melhor custo mínimo entre todas as iterações        |
| final_bitstring_distribution  | `CountsDict`              | O dicionário de contagens de bitstring associado ao custo mínimo        |
| solution_bitstring | `str`              | Bitstring com o melhor custo na distribuição final         |
| iteration_count  | `int`              | O número total de iterações do QAOA realizadas pelo Solver        |
| variables_to_bitstring_index_map  | `float`              | O mapeamento das variáveis para o bit equivalente na bitstring       |
| best_parameters  | `list[float]`            | Os parâmetros beta e gamma otimizados em todas as iterações  |
| warnings  |`list[str]`              | Os avisos produzidos durante a compilação ou execução do QAOA (padrão: None)   |
## Benchmarks
[Resultados de benchmarking publicados](https://arxiv.org/abs/2406.01743) mostram que o Solver resolve com sucesso problemas com mais de 120 qubits, superando até mesmo resultados previamente publicados em dispositivos de recozimento quântico e de íons aprisionados. As métricas de benchmark a seguir fornecem uma indicação aproximada da precisão e da escala dos tipos de problema com base em alguns exemplos. As métricas reais podem variar com base em diversas características do problema, como o número de termos na função objetivo (densidade) e sua localidade, o número de variáveis e a ordem polinomial.

O "Número de qubits" indicado não é uma limitação rígida, mas representa limites aproximados em que você pode esperar uma precisão de solução extremamente consistente. Problemas de maior escala já foram resolvidos com sucesso, e testes além desses limites são encorajados.

Conectividade arbitrária de qubits é suportada em todos os tipos de problema.

| Tipo de problema    | Número de qubits | Exemplo | Precisão | Tempo total (s) | Uso de runtime (s) | Número de iterações
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Problemas quadráticos com conexões esparsas  | 156 | Max-Cut 3-regular| 100%     | 1764     | 293          | 16 |
| Otimização binária de ordem superior | 156 | Modelo Ising spin-glass | 100%      | 1461     | 272          | 16 |
| Problemas quadráticos com conexões densas | 50 | Max-Cut totalmente conectado| 100%      |  1758    | 268  | 12 |
| Problema com restrições e termos de penalidade | 50 | Cobertura Mínima de Vértices Ponderada com 8% de densidade de arestas | 100%      | 1074     | 215 | 10 |
## Primeiros passos
Primeiro, autentique-se usando sua [chave de API do IBM Quantum](http://quantum.cloud.ibm.com/). Em seguida, selecione a Qiskit Function da seguinte forma. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [2]:
# %pip install networkx numpy

## Exemplo: Otimização sem restrições
Execute o problema de [corte máximo](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). O exemplo a seguir demonstra as capacidades do Solver em um problema Max-Cut em um grafo 3-regular não ponderado com 156 nós, mas você também pode resolver problemas com grafos ponderados.
Além do `qiskit-ibm-catalog`, você também usará os seguintes pacotes para executar este exemplo: `networkx` e `numpy`. Você pode instalar esses pacotes descomentando a célula a seguir se estiver executando este exemplo em um notebook com o kernel IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Saída da célula de código anterior](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

O Solver aceita uma string como entrada da definição do problema.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


Verifique o [status](/guides/functions#check-job-status) da carga de trabalho da sua Qiskit Function ou recupere os [resultados](/guides/functions#retrieve-results) da seguinte forma:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. Recuperar o resultado
Recupere o valor de corte ótimo do dicionário de resultados.

> **Note:** O mapeamento das variáveis para a bitstring pode ter mudado. O dicionário de saída contém um subdicionário `variables_to_bitstring_index_map`, que ajuda a verificar a ordenação.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Você pode verificar a precisão do resultado resolvendo o problema classicamente com solvers de código aberto como o [PuLP](https://coin-or.github.io/pulp/), caso o grafo não seja densamente conectado. Problemas de alta densidade podem exigir solvers clássicos avançados para validar a solução.
## Exemplo: Otimização com restrições
O exemplo anterior de Max-Cut é um problema comum de otimização binária quadrática sem restrições. O Optimization Solver do Q-CTRL pode ser usado para diversos tipos de problemas, incluindo otimização com restrições. Você pode resolver tipos de problema arbitrários fornecendo a definição do problema representada como um polinômio em que as restrições são modeladas como termos de penalidade.

O exemplo a seguir demonstra como construir uma função de custo para um problema de otimização com restrições, a [cobertura mínima de vértices](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Além dos pacotes `qiskit-ibm-catalog` e `qiskit`, você também usará os seguintes pacotes para executar este exemplo: `numpy`, `networkx` e `sympy`. Você pode instalar esses pacotes descomentando a célula a seguir se estiver executando este exemplo em um notebook com o kernel IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. Definir o problema
Defina um problema MVC aleatório gerando um grafo com nós ponderados aleatoriamente.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Saída da célula de código anterior](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

Um modelo de otimização padrão para MVC ponderado pode ser formulado da seguinte forma. Primeiro, uma penalidade deve ser adicionada para qualquer caso em que uma aresta não esteja conectada a um vértice no subconjunto. Portanto, seja $n_i = 1$ se o vértice $i$ está na cobertura (ou seja, no subconjunto) e $n_i = 0$ caso contrário. Em segundo lugar, o objetivo é minimizar o número total de vértices no subconjunto, o que pode ser representado pela seguinte função:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

Agora, cada aresta no grafo deve incluir pelo menos um ponto de extremidade da cobertura, o que pode ser expresso como a inequação:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Todo caso em que uma aresta não está conectada ao vértice de cobertura deve ser penalizado. Isso pode ser representado na função de custo adicionando uma penalidade da forma $P(1-n_i-n_j+n_i n_j)$, em que $P$ é uma constante de penalidade positiva. Assim, uma alternativa sem restrições para a inequação restrita para MVC ponderado é:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. Executar o problema